<a href="https://colab.research.google.com/github/shubhishukla8176-arch/exfoliation-energy-ml/blob/main/notebooks/04_structure_descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install jarvis-tools pymatgen

print("Required libraries installed!")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Required libraries installed!


In [2]:

import os
import json
import numpy as np
import pandas as pd

from google.colab import drive

from jarvis.core.atoms import Atoms
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.local_env import CrystalNN

print("All libraries imported successfully!")


All libraries imported successfully!


In [3]:

# Connect Google Drive
drive.mount('/content/drive')

# Project folder
project_folder = "/content/drive/MyDrive/MASC520_Project"

# Load original JARVIS data
json_path = os.path.join(
    project_folder,
    "exfoliation_energy_dataset.json"
)

with open(json_path, "r") as file:
    records = json.load(file)

print("Dataset loaded successfully!")
print("Total materials:", len(records))


Mounted at /content/drive
Dataset loaded successfully!
Total materials: 813


In [4]:

# Select the first material
first_record = records[0]

# Convert JARVIS dictionary into an Atoms object
first_atoms = Atoms.from_dict(first_record["atoms"])

# Convert it into a Pymatgen Structure
first_structure = first_atoms.pymatgen_converter()

print("Material ID:", first_record["jid"])

print("\nNumber of atoms:", first_atoms.num_atoms)

print("\nLattice parameters:")
print("a, b, c:", first_atoms.lattice.abc)

print("\nLattice angles:")
print("alpha, beta, gamma:", first_atoms.lattice.angles)

print("\nUnit cell volume:")
print(first_atoms.volume)

print("\nCrystal structure:")
print(first_structure)


Material ID: JVASP-10

Number of atoms: 3

Lattice parameters:
a, b, c: [3.355502, 3.355502, 6.22081]

Lattice angles:
alpha, beta, gamma: [np.float64(90.0), np.float64(90.0), np.float64(120.0)]

Unit cell volume:
60.658388812086834

Crystal structure:
Full Formula (V1 Se2)
Reduced Formula: VSe2
abc   :   3.355497   3.355497   6.220805
angles:  90.000000  90.000000 119.999997
pbc   :       True       True       True
Sites (3)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  V     0         0         0
  1  Se    0.666667  0.333333  0.747961
  2  Se    0.333333  0.666667  0.252039


In [5]:

structure_records = []
structure_failures = []

for i, record in enumerate(records):

    jid = record["jid"]

    try:
        # Convert JARVIS data into an Atoms object
        atoms = Atoms.from_dict(record["atoms"])

        # Extract lattice information
        a, b, c = atoms.lattice.abc
        alpha, beta, gamma = atoms.lattice.angles

        # Calculate volume and number of atoms
        volume = atoms.volume
        n_atoms = atoms.num_atoms

        # Volume per atom
        volume_per_atom = volume / n_atoms

        # Convert to Pymatgen structure
        structure = atoms.pymatgen_converter()

        # Store descriptors
        row = {
            "jid": jid,
            "number_of_atoms": n_atoms,
            "lattice_a": a,
            "lattice_b": b,
            "lattice_c": c,
            "alpha": alpha,
            "beta": beta,
            "gamma": gamma,
            "unit_cell_volume": volume,
            "volume_per_atom": volume_per_atom,
            "b_over_a": b / a,
            "c_over_a": c / a
        }

        structure_records.append(row)

    except Exception as e:

        structure_failures.append({
            "jid": jid,
            "stage": "geometric_descriptors",
            "error": str(e)
        })

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1} materials")

geometry_df = pd.DataFrame(structure_records)

print("\nGeometric descriptor calculation completed!")
print("Successfully processed:", len(geometry_df))
print("Failed materials:", len(structure_failures))

display(geometry_df.head())


Processed 100 materials
Processed 200 materials
Processed 300 materials
Processed 400 materials
Processed 500 materials
Processed 600 materials
Processed 700 materials
Processed 800 materials

Geometric descriptor calculation completed!
Successfully processed: 813
Failed materials: 0


,jid,number_of_atoms,lattice_a,lattice_b,lattice_c,alpha,beta,gamma,unit_cell_volume,volume_per_atom,b_over_a,c_over_a
0,JVASP-10,3,3.355502,3.355502,6.220810,90.000,90.0,120.0000,60.658389,20.219463,1.000000,1.853913
1,JVASP-60702,44,9.067080,9.601400,13.805400,90.000,90.0,90.0000,1201.850862,27.314792,1.058930,1.522585
2,JVASP-28397,8,3.958760,6.655930,23.940450,90.000,90.0,90.0000,630.812423,78.851553,1.681317,6.047462
3,JVASP-3534,6,3.927410,3.927410,7.402830,90.000,90.0,90.0000,114.185162,19.030860,1.000000,1.884914
4,JVASP-323,3,3.413340,3.413085,5.716881,90.016,90.0,120.0025,57.677405,19.225802,0.999925,1.674864


In [6]:

symmetry_records = []
symmetry_failures = []

for i, record in enumerate(records):

    jid = record["jid"]

    try:
        atoms = Atoms.from_dict(record["atoms"])
        structure = atoms.pymatgen_converter()

        # Analyze crystal symmetry
        analyzer = SpacegroupAnalyzer(
            structure,
            symprec=0.1,
            angle_tolerance=5
        )

        symmetry_records.append({
            "jid": jid,
            "spacegroup_number": analyzer.get_space_group_number(),
            "spacegroup_symbol": analyzer.get_space_group_symbol(),
            "crystal_system": analyzer.get_crystal_system()
        })

    except Exception as e:

        symmetry_failures.append({
            "jid": jid,
            "error": str(e)
        })

    if (i + 1) % 100 == 0:
        print(f"Analyzed {i + 1} materials")

symmetry_df = pd.DataFrame(symmetry_records)

print("\nSymmetry analysis completed!")
print("Successfully analyzed:", len(symmetry_df))
print("Failed materials:", len(symmetry_failures))

display(symmetry_df.head())


Analyzed 100 materials
Analyzed 200 materials
Analyzed 300 materials
Analyzed 400 materials
Analyzed 500 materials
Analyzed 600 materials
Analyzed 700 materials
Analyzed 800 materials

Symmetry analysis completed!
Successfully analyzed: 813
Failed materials: 0


,jid,spacegroup_number,spacegroup_symbol,crystal_system
0,JVASP-10,164,P-3m1,trigonal
1,JVASP-60702,61,Pbca,orthorhombic
2,JVASP-28397,53,Pmna,orthorhombic
3,JVASP-3534,129,P4/nmm,tetragonal
4,JVASP-323,164,P-3m1,trigonal


In [7]:

# Initialize the CrystalNN method
cnn = CrystalNN()

coordination_records = []
coordination_failures = []

for i, record in enumerate(records):

    jid = record["jid"]

    try:
        atoms = Atoms.from_dict(record["atoms"])
        structure = atoms.pymatgen_converter()

        coordination_numbers = []
        weighted_coordination = []
        nearest_distances = []

        # Analyze every atomic site
        for site_index in range(len(structure)):

            neighbors = cnn.get_nn_info(
                structure,
                site_index
            )

            coordination_numbers.append(len(neighbors))

            weighted_coordination.append(
                sum(
                    neighbor.get("weight", 1.0)
                    for neighbor in neighbors
                )
            )

            # Distances to the identified neighboring sites
            distances = [
                structure.get_distance(
                    site_index,
                    neighbor["site_index"]
                )
                for neighbor in neighbors
            ]

            if distances:
                nearest_distances.append(min(distances))

        coordination_records.append({
            "jid": jid,
            "coordination_mean": np.mean(coordination_numbers),
            "coordination_min": np.min(coordination_numbers),
            "coordination_max": np.max(coordination_numbers),
            "coordination_std": np.std(coordination_numbers),
            "weighted_coordination_mean": np.mean(weighted_coordination),
            "nearest_neighbor_distance_mean": np.mean(nearest_distances),
            "nearest_neighbor_distance_min": np.min(nearest_distances),
            "nearest_neighbor_distance_max": np.max(nearest_distances)
        })

    except Exception as e:

        coordination_failures.append({
            "jid": jid,
            "error": str(e)
        })

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1} materials")

coordination_df = pd.DataFrame(coordination_records)

print("\nCoordination calculations completed!")
print("Successfully processed:", len(coordination_df))
print("Failed materials:", len(coordination_failures))

display(coordination_df.head())


/usr/local/lib/python3.13/dist-packages/pymatgen/core/local_env.py:4236: UserWarning: No oxidation states specified on sites! For better results, set the site oxidation states in the structure.
  r1 = _get_radius(structure[n])
/usr/local/lib/python3.13/dist-packages/pymatgen/core/local_env.py:4238: UserWarning: No oxidation states specified on sites! For better results, set the site oxidation states in the structure.
  r2 = _get_radius(entry["site"])
/usr/local/lib/python3.13/dist-packages/pymatgen/core/local_env.py:4193: UserWarning: CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.
  return self._get_nn_data(structure, n, length, self._get_targets(structure, n))


Processed 50 materials
Processed 100 materials
Processed 150 materials
Processed 200 materials
Processed 250 materials
Processed 300 materials
Processed 350 materials
Processed 400 materials
Processed 450 materials
Processed 500 materials
Processed 550 materials
Processed 600 materials
Processed 650 materials
Processed 700 materials
Processed 750 materials
Processed 800 materials

Coordination calculations completed!
Successfully processed: 813
Failed materials: 0


,jid,coordination_mean,coordination_min,coordination_max,coordination_std,weighted_coordination_mean,nearest_neighbor_distance_mean,nearest_neighbor_distance_min,nearest_neighbor_distance_max
0,JVASP-10,4.000000,3,6,1.414214,4.000000,2.492265,2.492265,2.492265
1,JVASP-60702,2.181818,1,4,0.833196,2.181818,2.360655,2.336446,2.402193
2,JVASP-28397,2.000000,2,2,0.000000,2.000000,2.170313,2.170298,2.170328
3,JVASP-3534,4.000000,4,4,0.000000,4.000000,2.588065,2.341228,3.081739
4,JVASP-323,4.000000,3,6,1.414214,4.000000,2.428815,2.428815,2.428815


In [8]:

# Start with geometric descriptors
structure_features = geometry_df.copy()

# Merge symmetry descriptors
structure_features = structure_features.merge(
    symmetry_df,
    on="jid",
    how="left",
    validate="one_to_one"
)

# Merge coordination descriptors
structure_features = structure_features.merge(
    coordination_df,
    on="jid",
    how="left",
    validate="one_to_one"
)

# Add target property
target_df = pd.DataFrame({
    "jid": [record["jid"] for record in records],
    "exfoliation_energy": [
        record["exfoliation_energy"] for record in records
    ]
})

structure_features = structure_features.merge(
    target_df,
    on="jid",
    how="left",
    validate="one_to_one"
)

print("Final structure descriptor dataset created!")

print("Number of materials:", len(structure_features))
print("Number of columns:", len(structure_features.columns))

display(structure_features.head())


Final structure descriptor dataset created!
Number of materials: 813
Number of columns: 24


,jid,number_of_atoms,lattice_a,lattice_b,lattice_c,alpha,beta,gamma,unit_cell_volume,volume_per_atom,...,crystal_system,coordination_mean,coordination_min,coordination_max,coordination_std,weighted_coordination_mean,nearest_neighbor_distance_mean,nearest_neighbor_distance_min,nearest_neighbor_distance_max,exfoliation_energy
0,JVASP-10,3,3.355502,3.355502,6.220810,90.000,90.0,120.0000,60.658389,20.219463,...,trigonal,4.000000,3,6,1.414214,4.000000,2.492265,2.492265,2.492265,83.78
1,JVASP-60702,44,9.067080,9.601400,13.805400,90.000,90.0,90.0000,1201.850862,27.314792,...,orthorhombic,2.181818,1,4,0.833196,2.181818,2.360655,2.336446,2.402193,88.81
2,JVASP-28397,8,3.958760,6.655930,23.940450,90.000,90.0,90.0000,630.812423,78.851553,...,orthorhombic,2.000000,2,2,0.000000,2.000000,2.170313,2.170298,2.170328,0.03
3,JVASP-3534,6,3.927410,3.927410,7.402830,90.000,90.0,90.0000,114.185162,19.030860,...,tetragonal,4.000000,4,4,0.000000,4.000000,2.588065,2.341228,3.081739,71.05
4,JVASP-323,3,3.413340,3.413085,5.716881,90.016,90.0,120.0025,57.677405,19.225802,...,trigonal,4.000000,3,6,1.414214,4.000000,2.428815,2.428815,2.428815,89.57


In [9]:

print("Total materials:", len(structure_features))

print("\nDuplicate material IDs:")
print(structure_features["jid"].duplicated().sum())

print("\nMissing values per column:")
print(structure_features.isnull().sum())

numeric_data = structure_features.select_dtypes(
    include=[np.number]
)

print("\nTotal infinite values:")
print(np.isinf(numeric_data.to_numpy()).sum())

print("\nGeometry failures:", len(structure_failures))
print("Symmetry failures:", len(symmetry_failures))
print("Coordination failures:", len(coordination_failures))


Total materials: 813

Duplicate material IDs:
0

Missing values per column:
jid                               0
number_of_atoms                   0
lattice_a                         0
lattice_b                         0
lattice_c                         0
alpha                             0
beta                              0
gamma                             0
unit_cell_volume                  0
volume_per_atom                   0
b_over_a                          0
c_over_a                          0
spacegroup_number                 0
spacegroup_symbol                 0
crystal_system                    0
coordination_mean                 0
coordination_min                  0
coordination_max                  0
coordination_std                  0
weighted_coordination_mean        0
nearest_neighbor_distance_mean    0
nearest_neighbor_distance_min     0
nearest_neighbor_distance_max     0
exfoliation_energy                0
dtype: int64

Total infinite values:
0

Geometry failures: 0

In [10]:

structure_folder = os.path.join(
    project_folder,
    "structure_features"
)

os.makedirs(structure_folder, exist_ok=True)

# Save structural descriptors
structure_features.to_csv(
    os.path.join(
        structure_folder,
        "structure_descriptors.csv"
    ),
    index=False
)

# Save calculation failures separately
pd.DataFrame(structure_failures).to_csv(
    os.path.join(
        structure_folder,
        "geometry_failures.csv"
    ),
    index=False
)

pd.DataFrame(symmetry_failures).to_csv(
    os.path.join(
        structure_folder,
        "symmetry_failures.csv"
    ),
    index=False
)

pd.DataFrame(coordination_failures).to_csv(
    os.path.join(
        structure_folder,
        "coordination_failures.csv"
    ),
    index=False
)

print("Structural descriptors and failure reports saved!")


Structural descriptors and failure reports saved!
